# Engineering GenAI with MLflow: From Prototype to Production

## The question this notebook answers

> **"I changed the prompt. Is the app better now? Can I ship it?"**

In a GenAI app, the **prompt** decides most of the behavior. A few words added to a prompt can change the answers in ways that are hard to predict. It may fix one case and break another.

A prototype that works on a few examples cannot answer this question. To answer it, you need to connect three things:

- **Which prompt version** was used
- **What actually happened** inside each request
- **How good the results were**, measured the same way every time

MLflow connects these for you. In this notebook, we follow one small app through that whole journey.

<div class="mermaid">
flowchart LR
    P["Prompt version<br/>(Prompt Registry)"] --> T["Traces<br/>(what happened)"]
    T --> E["Evaluation<br/>(how good)"]
    E --> D{"Ship it?"}
    D -->|Yes| A["Switch alias<br/>to the new version"]
    D -->|No| P
    A --> M["Monitoring<br/>(still good?)"]
    M --> P
    style P fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
    style T fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
    style E fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
    style A fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
    style M fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>


## The story

| Stage | Step | What you do | Feature |
|---|---|---|---|
| Development | Step 1 | Build the app with a quick prototype prompt (v1) | Prompt Registry |
| Development | Step 2 | See what happens inside each request | Tracing |
| Development | Step 3 | Write prompt v2 that spells out the format and the rules | Prompt Registry |
| Offline evaluation | Step 4 | Evaluate v1 and v2 on the same test data | Evaluation, LLM judges |
| Offline evaluation | Step 5 | Read the results and decide | Evaluation runs |
| Release | Step 6 | Point production to v2, without changing code | Prompt aliases |
| Deployment | Step 7 | Deploy the app, with monitors registered first | Databricks Apps, monitoring |
| Online monitoring | Step 8 | Use the app and watch the quality of real requests | Monitoring |

## Requirements

- A [Databricks Free Edition](https://www.databricks.com/learn/free-edition) account (free, no credit card needed)
- Import this notebook into your workspace, attach **serverless** compute, and run the cells from top to bottom
- Steps 1 to 6 take about 10 minutes. Step 7 (creating and deploying the app) takes about 10 more minutes. The first monitoring results in Step 8 appear 15 to 30 minutes later

## Where things are stored

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">ℹ️</span>
    <div>
      <strong style="color: #0d47a1; font-size: 1.1em;">Two places in Databricks</strong>
      <p style="margin: 8px 0 0 0; color: #333;"><b>Unity Catalog</b> stores data and prompts, organized as <code>catalog.schema.name</code>. Our review table is <code>workspace.genai_demo.customer_reviews</code>, and our prompt is <code>workspace.genai_demo.review_analyzer</code>.<br><br>An <b>MLflow experiment</b> stores the records of your app. We use two: a development experiment for traces and evaluation runs (Steps 1 to 6), and a production experiment for the deployed app's traces and monitoring results (Steps 7 and 8). The production experiment stores its traces in Unity Catalog tables.</p>
    </div>
  </div>
</div>

# Step 0: Setup

First, install the latest MLflow. `%restart_python` restarts Python so that the new version is loaded.

In [ ]:
%pip install -U "mlflow[databricks]" databricks-openai databricks-sdk
%restart_python

Next, we set up the following:

1. A **schema** in Unity Catalog to store our table and prompt
2. An **MLflow experiment** to store traces, evaluation runs and monitors
3. A **client** to call an LLM hosted on Databricks
4. **Autologging**, so that every LLM call is traced without extra code

<div style="border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">📝</span>
    <div>
      <strong style="color: #4a148c; font-size: 1.1em;">Running the notebook again</strong>
      <p style="margin: 8px 0 0 0; color: #333;">
Every run adds traces and evaluation runs to the experiment, and every <code>register_prompt</code> adds a new prompt version.
After a few runs, the experiment is crowded, and v1 and v2 become version 5 and version 6.
<br><br>
To avoid this, the cell below <b>starts fresh</b> when <code>START_FRESH = True</code> (the default). Before setting things up, it deletes:
<ul>
<li>The development experiment, with its traces and evaluation runs</li>
<li>The prompt <code>review_analyzer</code>, with its aliases and versions</li>
</ul>
It keeps the production experiment, the app and the monitors (Steps 7 and 8), so the app's traces and monitor results stay.
On your first run, there is nothing to delete, so nothing happens.
Set <code>START_FRESH = False</code> if you want to keep the earlier results.
</p>
    </div>
  </div>
</div>

In [ ]:
import json
import os

import mlflow
from databricks_openai import DatabricksOpenAI

# --- 1. Unity Catalog schema ---
# Free Edition comes with a catalog named "workspace".
# We create a schema (a folder for tables and prompts) inside it.
CATALOG = "workspace"
SCHEMA = "genai_demo"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# The LLM to use. You can find available models on the "AI Gateway" page.
# This model is hosted by Databricks, so you don't need an API key.
MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

username = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_NAME = f"/Users/{username}/genai_prototype_to_production"
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.review_analyzer"

# --- Start fresh (see the box above) ---
# Deletes the development experiment and the prompt from earlier runs.
# The production experiment, the app and the monitors are not touched.
START_FRESH = True

if START_FRESH:
    import time

    from mlflow import MlflowClient

    mlflow_client = MlflowClient()

    # The development experiment. A deleted experiment keeps its name,
    # so we rename it first. This lets us create a new one with the same name below.
    old = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if old is not None:
        mlflow_client.rename_experiment(old.experiment_id, f"{EXPERIMENT_NAME}_deleted_{int(time.time())}")
        mlflow_client.delete_experiment(old.experiment_id)
        print("Deleted the development experiment from earlier runs")

    # The prompt. In Unity Catalog, you must delete the aliases and all versions first.
    # Until Step 1 registers v1 again, the app (if you deployed it) can't load the prompt.
    try:
        versions, token = [], None
        while True:
            page = mlflow_client.search_prompt_versions(PROMPT_NAME, page_token=token)
            versions.extend(page)
            token = page.token
            if not token:
                break
        for alias in ["production", "staging"]:
            try:
                mlflow.genai.delete_prompt_alias(name=PROMPT_NAME, alias=alias)
            except Exception:
                pass
        for version in versions:
            mlflow_client.delete_prompt_version(PROMPT_NAME, version.version)
        mlflow_client.delete_prompt(PROMPT_NAME)
        print(f"Deleted the prompt {PROMPT_NAME} ({len(versions)} versions)")
    except Exception as e:
        # On your first run there is no prompt yet, so this is expected
        print(f"No prompt deleted: {type(e).__name__}")

# --- 2. MLflow experiment ---
# An experiment is a container for the records of your app.
# We create one in your home folder.
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = experiment.experiment_id

# A helper that shows a link to the experiment page
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")


def experiment_link(label="Open the MLflow experiment"):
    displayHTML(f'<a href="https://{WORKSPACE_URL}/ml/experiments/{EXPERIMENT_ID}" target="_blank">{label}</a>')


# Free Edition has tighter rate limits on LLM calls.
# These settings make evaluation (Step 4) run one record at a time.
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"

# --- 3. LLM client ---
# DatabricksOpenAI is the OpenAI Python client, set up for Databricks.
# It connects to your workspace with your credentials automatically.
client = DatabricksOpenAI()

# --- 4. Autologging ---
# Every call made with the OpenAI client is recorded as part of a trace:
# the prompt, the response, token usage and latency.
mlflow.openai.autolog()

print("Setup complete")
experiment_link()

Our sample data is 30 customer reviews for a fictional bakery chain. Some are clearly positive or negative, some are just facts (like opening hours), and some have mixed feelings.

We save them as a table in Unity Catalog. In a real project, this table would come from your data pipeline.

In [ ]:
reviews = [
    "The croissants are amazing. Perfectly buttery and flaky every time.",
    "Terrible service. We waited 20 minutes and the staff was rude.",
    "Stopped by for a baguette. It was okay.",
    "Best sourdough in town, and the staff always greet you with a smile.",
    "The cake was dry and overpriced. Never coming back.",
    "Open 8 AM on weekdays and 9 AM on weekends.",
    "I love the smell of fresh bread and the cozy atmosphere.",
    "They got my order wrong two times in a row. Disappointing.",
    "Tried the famous cinnamon rolls and they lived up to the hype!",
    "The checkout line is way too long. Please add more staff at the register.",
    "It's a 5-minute walk from the station.",
    "The sandwich bread is so soft. My kids love it. We'll be back.",
    "The seating area is too small to relax.",
    "Tried the new pistachio danish. Not too sweet, just how I like it.",
    "The portions feel small for the price.",
    "Get the seasonal pumpkin muffin before it sells out.",
    "They accept cash and credit cards.",
    "The coffee is average, but the bread and coffee combo is a good deal.",
    "The bread tastes great, but the wait is always way too long.",
    "This used to be my favorite place, but the quality has gone down lately.",
    "You get what you pay for, I guess.",
    "Honestly, it wasn't as bad as the reviews said.",
    "The background music is relaxing. I always end up staying longer.",
    "The fried pastries tasted like old oil. Please fix this.",
    "Dogs are welcome on the terrace.",
    "The cinnamon roll is perfectly sweet and goes great with coffee.",
    "On weekends it's so crowded that sometimes you can't even get in.",
    "The new staff member was trying really hard. Loved the attitude.",
    "Bags cost extra, so bring your own.",
    "Everything tastes good. My only complaint is there's no parking.",
]

REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.customer_reviews"

# Create a Spark DataFrame with an ID and the review text, and save it as a table.
# mode("overwrite") replaces the table if you run this cell again.
(
    spark.createDataFrame([(i + 1, r) for i, r in enumerate(reviews)], "review_id INT, review STRING")
    .write.mode("overwrite")
    .saveAsTable(REVIEW_TABLE)
)
display(spark.table(REVIEW_TABLE).limit(5))

# Step 1 · Development: Build the app with prompt v1

We want our app to read one review and return this JSON, so that other systems (a dashboard, a ticket system) can use the results:

```json
{
  "sentiment": "positive | negative | neutral",
  "topics": ["product_quality", "service", "price", "atmosphere"],
  "actionable_feedback": {"has_feedback": true, "summary": "..."}
}
```

## Why register the prompt?

If you write the prompt directly in your code, every change to the prompt is a code change, and you lose track of which version produced which answer.

With **MLflow Prompt Registry**, the prompt is saved in Unity Catalog as its own asset:

- Every change becomes a new **version** (v1, v2, ...), with a message that says what changed
- You give a version an **alias**, like `production`. The app loads the prompt by alias, so you can switch versions without changing code (Step 6)
- When the app loads a prompt inside a trace, the **prompt version is linked to the trace**. You can always tell which version produced an answer (Step 2)

## v1: a quick prototype prompt

Most GenAI apps start with a short prompt written in a minute: "read this and give me JSON." Our v1 is exactly that. Let's register it and see how far it gets.

In [ ]:
# The prompt template. {{review}} is a placeholder for the review text.
# v1 is a quick prototype: one short instruction.
PROMPT_V1 = """You analyze customer reviews for a bakery chain.
Read the review below and return JSON with the sentiment, the topics, and any actionable feedback.

Review:
{{review}}
"""

# The prompt's name in Unity Catalog: catalog.schema.name
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.review_analyzer"

# Register v1. Each call to register_prompt creates a new version.
prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=PROMPT_V1,
    commit_message="v1: quick prototype prompt",
)

# Point the "production" alias to v1. The app always loads the "production" version.
mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="production", version=prompt_v1.version)

print(f"Registered {PROMPT_NAME} version {prompt_v1.version} as production")

Now we write the app. It does three things:

1. Load the prompt by alias from Prompt Registry
2. Put the review into the prompt and call the LLM
3. Convert the JSON text into a Python dictionary

In [ ]:
@mlflow.trace  # Record each call of this function as a trace (Step 2)
def analyze_review(review: str, alias: str = "production") -> dict:
    # 1. Load the prompt version that the alias points to.
    #    Because this runs inside a trace, the prompt version is linked to the trace.
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@{alias}")

    # 2. Put the review into the prompt and call the LLM
    response = client.chat.completions.create(
        model=MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt.template.replace("{{review}}", review)}],
        temperature=0.0,  # Always pick the most likely answer, so results are repeatable
        response_format={"type": "json_object"},  # Ask the model to return valid JSON
    )

    # 3. Convert the JSON text into a Python dictionary
    return json.loads(response.choices[0].message.content)


# Try it on one review with mixed feelings
sample_review = reviews[18]
print("Review:", sample_review)
print(json.dumps(analyze_review(sample_review), indent=2))

<div style="border-left: 4px solid #2e7d32; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">💡</span>
    <div>
      <strong style="color: #1b5e20; font-size: 1.1em;">Where to find the prompt</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Open the experiment, click <b>Prompts</b>, and select the schema <code>workspace.genai_demo</code>. You can see <code>review_analyzer</code> with its versions and aliases. Prompts are not shown in Catalog Explorer.</p>
    </div>
  </div>
</div>

# Step 2 · Development: See what happens inside each request

You just ran the app once. MLflow recorded it as a **trace**.

A trace is a record of one request. It is made of **spans**, one for each step inside the request:

- `@mlflow.trace` on `analyze_review` records the whole call: its inputs, outputs and latency
- `mlflow.openai.autolog()` (from Step 0) adds a span for the LLM call inside it: the full prompt, the raw response and token usage
- `load_prompt` inside the trace links the **prompt version** to the trace

<div class="mermaid">
flowchart TB
    subgraph T["Trace: one request"]
        V["Linked prompt version: review_analyzer v1"]
        subgraph A["Span: analyze_review, from @mlflow.trace"]
            I["inputs, outputs, latency"]
            L["Span: LLM call, from autolog<br/>prompt, response, token usage"]
        end
    end
    style T fill:#f7f5f0,stroke:#4a5568
    style A fill:#fdf0ea,stroke:#c2410c
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>


## What to check in the trace

Open the trace below the cell output of Step 1, or open the experiment and click **Traces**.

| Check | Why it matters |
|---|---|
| Inputs and outputs of `analyze_review` | Did the app return what you expected? |
| The full prompt in the LLM call span | Is the review inserted correctly? This is what the LLM actually saw. |
| Token usage and latency | How much does one request cost, and how long does it take? |
| The linked prompt version | Which version of the prompt produced this answer? |

When an answer looks wrong, the trace is where you start. You can see exactly what the LLM received and returned, instead of guessing.

In [ ]:
experiment_link("Open the experiment and click Traces")

<div style="border-left: 4px solid #2e7d32; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">💡</span>
    <div>
      <strong style="color: #1b5e20; font-size: 1.1em;">Try it</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Run <code>analyze_review</code> on a few other reviews from the <code>reviews</code> list. Which reviews are easy? Which ones look tricky? Keep the tricky ones in mind for Step 4.</p>
    </div>
  </div>
</div>

In [ ]:
for review in [reviews[5], reviews[20], reviews[21]]:
    print(review)
    print(json.dumps(analyze_review(review)), "\n")

# Step 3 · Development: Write prompt v2

Look at v1's outputs in Steps 1 and 2. A quick prototype prompt usually has these problems:

- **The format is not fixed.** v1 never says which keys and values to use, so the model chooses. The key names, the sentiment labels and the topic words can change from review to review. A dashboard or another system that reads the JSON breaks.
- **Mixed reviews are guesswork.** Is "The bread tastes great, but the wait is always way too long" positive, negative or neutral? People can disagree, and so can the LLM.

For the second problem, our team decides on a rule: **in a mixed review, the main point decides the sentiment, and a complaint after "but" is usually the main point.** So this review is **negative**. The test data in Step 4 uses the same rule.

v2 makes everything we expect explicit:

- **The output format**: the exact keys, and the allowed values for sentiment and topics
- **Rules** for mixed reviews and fact-only reviews
- **Examples** that follow the format and the rules (few-shot prompting)

We register this as **v2** and give it the alias `staging`. Production still uses v1. Nothing changes for users until we decide to switch.

<div style="border-left: 4px solid #ef6c00; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">⚠️</span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">Examples must follow your rules</strong>
      <p style="margin: 8px 0 0 0; color: #333;">An example that contradicts your rules teaches the model the wrong thing. For instance, an example that labels “great bread, but a long wait” as positive pushes the model away from our rule, and the scores get worse. Also keep examples different from the test data, or the evaluation can’t tell you if the prompt works on new reviews.</p>
    </div>
  </div>
</div>

In [ ]:
# v2 spells out the output format, the rules and two examples
PROMPT_V2 = """You are an assistant that analyzes customer reviews for a bakery chain.
Extract the sentiment, topics and actionable feedback from the review below.

Return only JSON that follows this schema:
{
  "sentiment": "one of positive, negative, neutral",
  "topics": ["any of product_quality, service, price, atmosphere"],
  "actionable_feedback": {
    "has_feedback": true or false,
    "summary": "a short summary of the feedback (empty string if none)"
  }
}

Rules for sentiment:
- If a review has both praise and a complaint, decide by the main point. A complaint after "but" is usually the main point.
- Facts with no opinion, like opening hours or payment methods, are neutral.

Example 1:
Review: The coffee is great, but the staff ignored us for ten minutes.
Output: {"sentiment": "negative", "topics": ["product_quality", "service"], "actionable_feedback": {"has_feedback": true, "summary": "Greet and serve customers faster"}}

Example 2:
Review: We open at 8 AM.
Output: {"sentiment": "neutral", "topics": [], "actionable_feedback": {"has_feedback": false, "summary": ""}}

Review:
{{review}}
"""

prompt_v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=PROMPT_V2,
    commit_message="v2: define the output format, add rules for mixed reviews and two examples",
)
mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="staging", version=prompt_v2.version)

print(f"Registered version {prompt_v2.version} as staging")

# Try v2 on the same review as Step 1
print(json.dumps(analyze_review(sample_review, alias="staging"), indent=2))

Compare this output with v1's output in Step 1. Even if it looks better, one example can't tell you that v2 is better overall. That's what Step 4 is for.

# Step 4 · Offline evaluation: Evaluate v1 and v2 on the same test data

To compare two versions fairly, run both on the **same test data** and score them in the **same way**.

## Test data

We use 8 reviews with known answers. Each record has:

- `inputs`: what we pass to the app (the review)
- `expectations`: the correct answer (the expected sentiment)

The first 4 are clear cases. The last 4 are tricky, with mixed or subtle sentiment. If the data has only easy cases, both versions get everything right and you can't see a difference.

<div style="border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">📝</span>
    <div>
      <strong style="color: #4a148c; font-size: 1.1em;">In a real project</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Use 30 or more records, and add new ones whenever you find a case the app gets wrong. You can also save the test data as an evaluation dataset in Unity Catalog, so your team can reuse it.</p>
    </div>
  </div>
</div>

In [ ]:
eval_data = [
    # Clear cases
    {"inputs": {"review": "The croissants are amazing. Perfectly buttery and flaky every time."},
     "expectations": {"expected_sentiment": "positive"}},
    {"inputs": {"review": "Terrible service. We waited 20 minutes and the staff was rude."},
     "expectations": {"expected_sentiment": "negative"}},
    {"inputs": {"review": "Stopped by for a baguette. It was okay."},
     "expectations": {"expected_sentiment": "neutral"}},
    {"inputs": {"review": "I love the smell of fresh bread and the cozy atmosphere."},
     "expectations": {"expected_sentiment": "positive"}},
    # Tricky cases
    {"inputs": {"review": "The bread tastes great, but the wait is always way too long."},
     "expectations": {"expected_sentiment": "negative"}},
    {"inputs": {"review": "You get what you pay for, I guess."},
     "expectations": {"expected_sentiment": "neutral"}},
    {"inputs": {"review": "Honestly, it wasn't as bad as the reviews said."},
     "expectations": {"expected_sentiment": "positive"}},
    {"inputs": {"review": "This used to be my favorite place, but the quality has gone down lately."},
     "expectations": {"expected_sentiment": "negative"}},
]

## LLM judges

Reading every output by hand doesn't scale. An **LLM judge** uses an LLM to score each output against criteria you write in plain English. Each judge also writes a short **rationale** that explains its decision.

We create three judges with `make_judge`. Each one checks one thing and returns `pass` or `fail`.

| Judge | What it checks | Needs the expected answer? |
|---|---|---|
| `sentiment_correctness` | Does the sentiment match the expected answer? | Yes |
| `schema_compliance` | Does the output follow the JSON schema? | No |
| `topics_relevance` | Do the topics match what the review talks about? | No |

In the instructions, you can use these placeholders. MLflow fills them in for each record:

- `{{ inputs }}`: the input passed to the app
- `{{ outputs }}`: what the app returned
- `{{ expectations }}`: the expected answer from the test data

In [ ]:
from typing import Literal

from mlflow.genai.judges import make_judge

# The LLM used by the judges. Here we use the same model as the app.
JUDGE_MODEL = f"databricks:/{MODEL_ENDPOINT}"

sentiment_judge = make_judge(
    name="sentiment_correctness",
    instructions=(
        "Compare the sentiment field in {{ outputs }} with expected_sentiment in {{ expectations }}. "
        "Return 'pass' if they match, otherwise 'fail'."
    ),
    model=JUDGE_MODEL,
    feedback_value_type=Literal["pass", "fail"],  # The judge can only answer "pass" or "fail"
)

schema_judge = make_judge(
    name="schema_compliance",
    instructions=(
        "Check whether {{ outputs }} meets all of these conditions: "
        "(1) sentiment is one of positive, negative, neutral; "
        "(2) topics is a list that contains only product_quality, service, price, atmosphere; "
        "(3) actionable_feedback has has_feedback (boolean) and summary (string). "
        "Return 'pass' if all are met, otherwise 'fail'."
    ),
    model=JUDGE_MODEL,
    feedback_value_type=Literal["pass", "fail"],
)

topics_judge = make_judge(
    name="topics_relevance",
    instructions=(
        "Read the review in {{ inputs }} and check whether the topics in {{ outputs }} "
        "match what the review actually talks about. "
        "Return 'pass' if they match, or 'fail' if there are unrelated or clearly missing topics."
    ),
    model=JUDGE_MODEL,
    feedback_value_type=Literal["pass", "fail"],
)

judges = [sentiment_judge, schema_judge, topics_judge]

## Run the evaluation

`mlflow.genai.evaluate` does the following for each record in the test data:

<div class="mermaid">
flowchart LR
    D["Test data<br/>inputs + expectations"] --> P["predict_fn<br/>runs the app"]
    P --> O["Output<br/>+ trace with prompt version"]
    O --> J["Judges<br/>pass or fail + rationale"]
    D -.->|expectations| J
    J --> R["Evaluation run<br/>in the experiment"]
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>


We run it once for each version. `predict_v1` uses the `production` alias (v1), and `predict_v2` uses the `staging` alias (v2). Each trace in the evaluation is linked to the prompt version it used, so **every score can be traced back to a prompt version**.

`predict_v1` and `predict_v2` also have `@mlflow.trace`, with the same span name `predict`. This makes each prediction one trace with the same structure for both versions, so MLflow can line up the results record by record and show the differences span by span.

<div style="border-left: 4px solid #ef6c00; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">⚠️</span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">This takes a few minutes</strong>
      <p style="margin: 8px 0 0 0; color: #333;">If you see a rate limit error, wait a minute and run the cell again.</p>
    </div>
  </div>
</div>

In [ ]:
# Both functions use the same span name, "predict". The comparison view matches
# spans by name, so it can show the differences between v1 and v2 side by side.
@mlflow.trace(name="predict")
def predict_v1(review):
    return analyze_review(review, alias="production")  # v1


@mlflow.trace(name="predict")
def predict_v2(review):
    return analyze_review(review, alias="staging")  # v2


# Each evaluation is saved as an "evaluation run" in the experiment
with mlflow.start_run(run_name="eval_v1"):
    result_v1 = mlflow.genai.evaluate(data=eval_data, predict_fn=predict_v1, scorers=judges)

with mlflow.start_run(run_name="eval_v2"):
    result_v2 = mlflow.genai.evaluate(data=eval_data, predict_fn=predict_v2, scorers=judges)

# Step 5 · Offline evaluation: Read the results and decide

An evaluation gives you numbers, but numbers alone don't tell you what to do. Read the results in this order:

| | Question | Where to look |
|---|---|---|
| 1 | Overall, which version scores higher on each judge? | The pass counts below |
| 2 | Which reviews got different results between v1 and v2? | The table below, and the comparison view in the experiment |
| 3 | Why did they get different results? | The trace and the judge's rationale for each review |
| 4 | Ship v2, or improve the prompt again? | Your decision, based on 1 to 3 |

## 1. Compare the overall scores

Each judge's result is saved on the trace of each review. We collect them from the two evaluation runs and count how many reviews passed.

In [ ]:
import pandas as pd

JUDGE_NAMES = [j.name for j in judges]


def judge_results(run_id):
    """Return {review: {judge name: "pass" or "fail"}} for one evaluation run."""
    results = {}
    for trace in mlflow.search_traces(run_id=run_id, return_type="list"):
        # The app's input (the review) is saved on the trace as JSON
        try:
            review = json.loads(trace.data.request).get("review", trace.data.request)
        except Exception:
            review = str(trace.data.request)
        # The judges' results are saved on the trace as assessments
        for a in trace.info.assessments:
            if a.name in JUDGE_NAMES:
                value = a.feedback.value if getattr(a, "feedback", None) else getattr(a, "value", None)
                results.setdefault(review, {})[a.name] = str(value).lower()
    return results


v1_results = judge_results(result_v1.run_id)
v2_results = judge_results(result_v2.run_id)


def pass_count(results, judge):
    values = [r.get(judge) for r in results.values()]
    return f"{values.count('pass')} / {len(values)}"


summary = pd.DataFrame({
    "judge": JUDGE_NAMES,
    "v1 passed": [pass_count(v1_results, j) for j in JUDGE_NAMES],
    "v2 passed": [pass_count(v2_results, j) for j in JUDGE_NAMES],
})
display(summary)

With only 8 reviews, **one review changes the score by 12.5 points**. A small difference may just be one review. Don't decide from the summary alone. Look at the reviews behind the numbers.

## 2. Find the reviews where v1 and v2 disagree

The table below shows each review's result for both versions. `changed` is `True` when at least one judge gave a different result.

In [ ]:
rows = []
for review in v1_results:
    row = {"review": review}
    for j in JUDGE_NAMES:
        row[f"{j} (v1)"] = v1_results[review].get(j)
        row[f"{j} (v2)"] = v2_results.get(review, {}).get(j)
    row["changed"] = any(row[f"{j} (v1)"] != row[f"{j} (v2)"] for j in JUDGE_NAMES)
    rows.append(row)

comparison = pd.DataFrame(rows).sort_values("changed", ascending=False)
display(comparison)

To see the outputs and the judges' rationale for these reviews, use the comparison view in the experiment:

1. Open the experiment and click **Evaluation runs**
2. Check the boxes for `eval_v1` and `eval_v2`
3. From the **Actions** menu, select **Compare**

You can see the outputs of both versions and the judges' results side by side, for each review. Look for reviews where one version passed and the other failed.

## 3. Find out why

For each review that changed, open the trace and read:

- **The output.** What did the app actually return?
- **The judge's rationale.** Why did the judge say pass or fail?

Judges are LLMs too, and **they can be wrong**. If you disagree with a judge's rationale, the problem is the judge, not the app. Improve the judge's instructions, or add human feedback to the trace (see Next steps).

## 4. Decide

| What you see | What to do |
|---|---|
| v2 is better on the judges you care about, and any drop on other judges has a reason you accept | Ship v2 (Step 6) |
| v2 fixes some reviews but breaks others | Write v3 that keeps the fixes, register it, and evaluate again |
| No real difference | The examples don't help for this data. Try a different change |

A score that goes down is not always bad news, so check why before you decide. For example, a judge may give a pass to free-form topic words like "wait time" because they match the review, even though your format only allows four topics. Read the rationale and decide whether the app or the judge is wrong.

Whatever the result is, you now **know** it before your users see it.

<div style="border-left: 4px solid #c2410c; background: #fdf0ea; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">🎯</span>
    <div>
      <strong style="color: #9a3412; font-size: 1.1em;">Why this matters</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Every score is linked to an evaluation run, every evaluation run to its traces, and every trace to the prompt version that produced it. So “Is v2 better?” has an answer you can check, explain and share with your team, instead of “it looked fine on a few examples.”</p>
    </div>
  </div>
</div>

In [ ]:
experiment_link("Open the experiment, click Evaluation runs, and compare eval_v1 and eval_v2")

# Step 6 · Release: Point production to v2

The app always loads the prompt with the `production` alias. To release v2, we move the alias from v1 to v2.

- **No code change.** The same `analyze_review` code now uses v2.
- **Rollback is just as easy.** Move the alias back to v1.
- **The history stays.** Traces from before the switch are linked to v1, and traces after it are linked to v2.

In Step 7, we deploy the app. It loads the prompt by the same alias, so it starts with v2. After that, releasing a new prompt version is just moving the alias. You don't need to redeploy the app.

<div style="border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">📝</span>
    <div>
      <strong style="color: #4a148c; font-size: 1.1em;">In this notebook</strong>
      <p style="margin: 8px 0 0 0; color: #333;">We switch to v2 because Step 5 told us v2 is better. After this step, <code>production</code> points to v2, so running Step 4 again would evaluate v2 twice. To start over, run the notebook from Step 1.</p>
    </div>
  </div>
</div>

In [ ]:
# Move the "production" alias to v2
mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="production", version=prompt_v2.version)
print(f"production now points to version {prompt_v2.version}")

# The same code, with the same alias, now uses v2
print(json.dumps(analyze_review(sample_review), indent=2))

# To roll back, move the alias back to v1:
# mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="production", version=prompt_v1.version)

Open the newest trace and check the linked prompt version. It is now v2, even though `analyze_review` didn't change.

In [ ]:
experiment_link("Open the experiment, click Traces, and check the prompt version of the newest trace")

# Step 7 · Deployment: Deploy the app with monitoring

So far, the app has been a function in this notebook. Now we deploy it as a web app with **Databricks Apps**, so that users can use it.

<div class="mermaid">
flowchart LR
    U["Users"] --> APP["Databricks App<br/>(review analyzer)"]
    APP -->|"load prompt @production"| PR["Prompt Registry<br/>v1, v2"]
    APP -->|traces| PX["Production experiment<br/>(traces in Unity Catalog tables)"]
    MON["Monitors<br/>schema_compliance, topics_relevance"] -->|"score new traces<br/>every ~15 min"| PX
    style APP fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
    style MON fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>


We do it in this order:

1. Create a **production experiment** that stores the app's traces in Unity Catalog, and register the **monitors** to it
2. Write the app's code
3. Create the app, and give it access to the LLM, the experiment and the prompt
4. Deploy the app

<div style="border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">📝</span>
    <div>
      <strong style="color: #4a148c; font-size: 1.1em;">Why a separate experiment for production?</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Evaluation runs and test traces stay in the development experiment. Real user traffic goes to the production experiment, and the monitors score only that traffic.</p>
    </div>
  </div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">ℹ️</span>
    <div>
      <strong style="color: #0d47a1; font-size: 1.1em;">Where the app's traces are stored</strong>
      <p style="margin: 8px 0 0 0; color: #333;">By default, an experiment keeps trace data in its own storage. For production, we store the traces in <b>Unity Catalog tables</b> instead. This is the recommended setup: the traces become Delta tables that you can query with SQL and control with Unity Catalog permissions.<br><br>On Free Edition, it is also required: a Databricks App can't upload trace data to the experiment's default storage, so the traces would have no content and the monitors could not score them.</p>
    </div>
  </div>
</div>


## 7-1. Production experiment and monitors

First, we create the production experiment with `trace_location=UnityCatalog(...)`, and save on it the SQL warehouse that the monitors use to read the traces. This creates four tables in `workspace.genai_demo` for the traces. Their names start with `review_app_otel_`, and the spans (the content of each trace) go to `review_app_otel_spans`.

An experiment can be linked to Unity Catalog only when it is created. If you have a production experiment from an older version of this notebook, the cell below replaces it.

Monitoring is attached to an **experiment**, not to the app. Any trace that arrives in the experiment is scored, wherever the app runs.

We reuse two judges from Step 4 as monitors. Both check the output without an expected answer, so they also work on real user traffic. `sentiment_correctness` needs an expected answer, so it can't be used here.

| Monitor | Checks |
|---|---|
| `schema_compliance` | Does the output still follow the JSON format? |
| `topics_relevance` | Do the topics match what the review talks about? |

<div style="border-left: 4px solid #2e7d32; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">💡</span>
    <div>
      <strong style="color: #1b5e20; font-size: 1.1em;">Register monitors before you deploy</strong>
      <p style="margin: 8px 0 0 0; color: #333;">If you register and start the monitors first, the monitoring job starts right away. We also pass the experiment ID explicitly, so the monitors are attached to the production experiment.</p>
    </div>
  </div>
</div>

In [ ]:
import time

from databricks.sdk import WorkspaceClient
from mlflow import MlflowClient
from mlflow.entities.trace_location import UnityCatalog
from mlflow.genai.scorers import ScorerSamplingConfig, list_scorers
from mlflow.tracing import set_databricks_monitoring_sql_warehouse_id

w = WorkspaceClient()

# Traces in Unity Catalog are read through a SQL warehouse.
# Free Edition has one warehouse, so we use the first one we find.
SQL_WAREHOUSE_ID = next(iter(w.warehouses.list())).id
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = SQL_WAREHOUSE_ID

PROD_EXPERIMENT_NAME = f"/Users/{username}/genai_review_app_production"
TRACE_TABLE_PREFIX = "review_app"


def linked_to_uc(exp):
    location = getattr(exp, "trace_location", None)
    return getattr(location, "full_otel_spans_table_name", None) is not None


# An experiment can be linked to Unity Catalog only when it is created.
# If there is an older production experiment without the link, rename it and delete it.
existing = mlflow.get_experiment_by_name(PROD_EXPERIMENT_NAME)
if existing is not None and not linked_to_uc(existing):
    MlflowClient().rename_experiment(existing.experiment_id, f"{PROD_EXPERIMENT_NAME}_deleted_{int(time.time())}")
    MlflowClient().delete_experiment(existing.experiment_id)
    print("Replaced an older production experiment that was not linked to Unity Catalog")

# The production experiment. Its traces are stored in Unity Catalog tables.
prod_experiment = mlflow.set_experiment(
    PROD_EXPERIMENT_NAME,
    trace_location=UnityCatalog(catalog_name=CATALOG, schema_name=SCHEMA, table_prefix=TRACE_TABLE_PREFIX),
)
PROD_EXPERIMENT_ID = prod_experiment.experiment_id

# The four tables for the traces. The app needs MODIFY and SELECT on them (Step 7-3).
TRACE_TABLES = [
    f"{CATALOG}.{SCHEMA}.{TRACE_TABLE_PREFIX}_otel_{kind}" for kind in ["spans", "logs", "metrics", "annotations"]
]
print("The app's spans go to:", prod_experiment.trace_location.full_otel_spans_table_name)

# The monitors read the traces from Unity Catalog through a SQL warehouse.
# The environment variable above works only in this notebook, so we save the warehouse ID
# on the experiment. Do this BEFORE registering the monitors, or the monitoring job fails.
set_databricks_monitoring_sql_warehouse_id(sql_warehouse_id=SQL_WAREHOUSE_ID, experiment_id=PROD_EXPERIMENT_ID)

# Switch back to the development experiment for the rest of this notebook
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

# Register the two judges from Step 4 to the production experiment and start them
already_registered = [s.name for s in list_scorers(experiment_id=PROD_EXPERIMENT_ID)]
for judge in [schema_judge, topics_judge]:
    if judge.name in already_registered:
        print(f"{judge.name} is already registered")
        continue
    monitor = judge.register(name=judge.name, experiment_id=PROD_EXPERIMENT_ID)
    monitor.start(
        experiment_id=PROD_EXPERIMENT_ID,
        sampling_config=ScorerSamplingConfig(sample_rate=1.0),  # Score every trace. In production, 0.1 to 0.3 is common
    )
    print(f"Started monitoring with {judge.name}")

## 7-2. The app's code

The app is a small [Streamlit](https://streamlit.io/) web page with a text box. It uses the same logic as `analyze_review` in this notebook:

- It loads the prompt by the `production` alias, inside a trace
- It sends each trace to the production experiment

We write three files to a folder in your workspace:

| File | What it does |
|---|---|
| `app.py` | The web page and the review analysis |
| `app.yaml` | How to start the app, and its settings (environment variables) |
| `requirements.txt` | The Python packages the app needs |

The app has sample buttons that put a review in the text box, and an **Analyze all samples** button that sends every sample at once. Each analysis becomes one trace in the production experiment, so this is a quick way to give the monitors (Step 8) something to score.

In [ ]:
APP_DIR = f"/Workspace/Users/{username}/genai_review_app"
os.makedirs(APP_DIR, exist_ok=True)

APP_PY = """import json
import os

import mlflow
import streamlit as st
from databricks_openai import DatabricksOpenAI

PROMPT_NAME = os.environ["PROMPT_NAME"]
MODEL_ENDPOINT = os.environ["MODEL_ENDPOINT"]

# Send traces to the production experiment (set by the "experiment" resource)
mlflow.set_experiment(experiment_id=os.environ["MLFLOW_EXPERIMENT_ID"])
mlflow.openai.autolog()

client = DatabricksOpenAI()


@mlflow.trace
def analyze_review(review: str) -> dict:
    # Load the prompt by alias. The prompt version is linked to the trace.
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    response = client.chat.completions.create(
        model=MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt.template.replace("{{review}}", review)}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


# Sample reviews. Click one to put it in the text box.
SAMPLES = {
    "Positive": "The croissants are perfect, and the staff always remembers my order.",
    "Negative": "My cake was stale, and nobody answered when I called to complain.",
    "Mixed": "The bread tastes great, but the wait is always way too long.",
    "Mixed (praise last)": "Parking is hard to find, but the cinnamon rolls are worth it.",
    "Fact only": "The new store on Main Street opens at 7 AM on weekdays.",
    "Lukewarm": "The coffee was fine. Nothing special, nothing bad.",
}

st.title("Bakery review analyzer")
version = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production").version
st.caption(f"Prompt: {PROMPT_NAME} (production = version {version})")

if "review" not in st.session_state:
    st.session_state["review"] = SAMPLES["Mixed"]


def use_sample(text):
    st.session_state["review"] = text


st.write("Try a sample:")
columns = st.columns(3)
for i, (label, text) in enumerate(SAMPLES.items()):
    columns[i % 3].button(label, on_click=use_sample, args=(text,), use_container_width=True)

review = st.text_area("Customer review", key="review")

left, right = st.columns(2)
if left.button("Analyze", type="primary", use_container_width=True):
    with st.spinner("Analyzing..."):
        st.json(analyze_review(review))

# Analyze every sample at once. Each one becomes its own trace,
# so this is a quick way to give the monitors something to score.
if right.button(f"Analyze all {len(SAMPLES)} samples", use_container_width=True):
    progress = st.progress(0.0)
    for i, (label, text) in enumerate(SAMPLES.items()):
        result = analyze_review(text)
        with st.expander(f"{label}: {result.get('sentiment', '?')}"):
            st.write(text)
            st.json(result)
        progress.progress((i + 1) / len(SAMPLES))
    st.success(f"Done. {len(SAMPLES)} traces were sent to the production experiment.")
"""

APP_YAML = f"""command: ["streamlit", "run", "app.py"]
env:
  - name: MLFLOW_EXPERIMENT_ID
    valueFrom: experiment
  - name: MLFLOW_TRACKING_URI
    value: "databricks"
  - name: MLFLOW_REGISTRY_URI
    value: "databricks-uc"
  - name: PROMPT_NAME
    value: "{PROMPT_NAME}"
  - name: MODEL_ENDPOINT
    value: "{MODEL_ENDPOINT}"
"""

REQUIREMENTS = """mlflow[databricks]>=3.14
databricks-openai
streamlit
"""

for file_name, content in [("app.py", APP_PY), ("app.yaml", APP_YAML), ("requirements.txt", REQUIREMENTS)]:
    with open(f"{APP_DIR}/{file_name}", "w") as f:
        f.write(content)

print(f"Wrote the app files to {APP_DIR}")

## 7-3. Create the app and give it access

A Databricks App runs as its own **service principal** (an identity for the app). It needs access to four things:

| What | How we give access |
|---|---|
| The LLM | App resource: serving endpoint, `CAN_QUERY` |
| The production experiment | App resource: MLflow experiment, `CAN_EDIT`. Its ID is passed to the app as `MLFLOW_EXPERIMENT_ID` (see `valueFrom: experiment` in `app.yaml`) |
| The prompt in Unity Catalog | `GRANT` on the catalog and the schema |
| The trace tables in Unity Catalog | `GRANT MODIFY, SELECT` on each of the four tables. `ALL PRIVILEGES` is not enough, so we grant them explicitly |

Creating the app for the first time takes a few minutes.

<div style="border-left: 4px solid #ef6c00; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">⚠️</span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">Why the app needs MANAGE</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Prompt Registry is in Beta. During the Beta, loading a prompt needs <code>CREATE FUNCTION</code>, <code>EXECUTE</code> and <code>MANAGE</code> on the schema, even if the app only reads it. <code>MANAGE</code> is a strong privilege: it lets the app manage permissions on the schema and the objects in it, including the review table in this notebook. In a real project, keep prompts in their own schema and give these privileges only there.</p>
    </div>
  </div>
</div>

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.apps import (
    App,
    AppResource,
    AppResourceExperiment,
    AppResourceExperimentExperimentPermission,
    AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission,
)

w = WorkspaceClient()
APP_NAME = "genai-review-analyzer"  # Lowercase letters, numbers and hyphens only

resources = [
    AppResource(
        name="experiment",
        experiment=AppResourceExperiment(
            experiment_id=PROD_EXPERIMENT_ID,
            permission=AppResourceExperimentExperimentPermission.CAN_EDIT,
        ),
    ),
    AppResource(
        name="model",
        serving_endpoint=AppResourceServingEndpoint(
            name=MODEL_ENDPOINT,
            permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY,
        ),
    ),
]

try:
    w.apps.get(APP_NAME)
    # The app exists. Update its resources, in case the production experiment was replaced.
    app = w.apps.update(name=APP_NAME, app=App(name=APP_NAME, description="Bakery review analyzer", resources=resources))
    print(f"The app {APP_NAME} already exists. Updated its resources")
except NotFound:
    print(f"Creating the app {APP_NAME}. This takes a few minutes...")
    app = w.apps.create_and_wait(App(name=APP_NAME, description="Bakery review analyzer", resources=resources))

# Let the app's service principal load the prompt.
# During the Prompt Registry Beta, this needs CREATE FUNCTION, EXECUTE and MANAGE on the schema,
# even for read-only use. In a real project, keep prompts in their own schema.
sp = app.service_principal_client_id
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{sp}`")
spark.sql(f"GRANT USE SCHEMA, CREATE FUNCTION, EXECUTE, MANAGE ON SCHEMA {CATALOG}.{SCHEMA} TO `{sp}`")
print(f"Granted access to {CATALOG}.{SCHEMA} for the app's service principal")

# Let the app write traces to the Unity Catalog tables
for table in TRACE_TABLES:
    spark.sql(f"GRANT MODIFY, SELECT ON TABLE {table} TO `{sp}`")
print("Granted MODIFY and SELECT on the trace tables")

## 7-4. Deploy the app

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">ℹ️</span>
    <div>
      <strong style="color: #0d47a1; font-size: 1.1em;">Apps on Free Edition</strong>
      <p style="margin: 8px 0 0 0; color: #333;">You can create up to 3 apps. An app stops automatically 24 hours after it starts or is deployed. If it has stopped, this cell starts it again before deploying.</p>
    </div>
  </div>
</div>

In [ ]:
from databricks.sdk.service.apps import AppDeployment

# Start the app's compute if it has stopped
state = w.apps.get(APP_NAME).compute_status.state
if state is not None and state.value != "ACTIVE":
    print("Starting the app...")
    w.apps.start_and_wait(APP_NAME)

print("Deploying the app. This takes a few minutes...")
deployment = w.apps.deploy_and_wait(APP_NAME, AppDeployment(source_code_path=APP_DIR))
print("Deployment status:", deployment.status.state.value)

app_url = w.apps.get(APP_NAME).url
displayHTML(f'<a href="{app_url}" target="_blank">Open the app</a>')

# Step 8 · Online monitoring: Use the app and watch the quality

Open the app with the link above and analyze a few reviews:

- Click a sample button (Positive, Mixed, Fact only and so on), then **Analyze**
- Or click **Analyze all samples** to send all six at once
- Or write your own review

Each request becomes a trace in the production experiment, linked to the prompt version it used. The trace is stored in the Unity Catalog tables from Step 7-1.

| | Offline evaluation (Steps 4 and 5) | Online monitoring (Step 8) |
|---|---|---|
| When | Before release, when you run it | After release, automatically, about every 15 minutes |
| Data | Test data with expected answers | Real requests from users |
| Scorers | Any scorer | Scorers that don't need expected answers |

Because each trace is linked to a prompt version, a monitoring result tells you **which prompt version** had the problem. That closes the loop: a problem in production leads back to the prompt, and the fix goes through Steps 3 to 6 again. To release the fix, you only move the `production` alias. The app picks up the new version within about a minute, because MLflow caches the alias for 60 seconds.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">ℹ️</span>
    <div>
      <strong style="color: #0d47a1; font-size: 1.1em;">Results take 15 to 30 minutes</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Open the production experiment and click <b>Traces</b>. When the monitors have run, you can see the <code>schema_compliance</code> and <code>topics_relevance</code> results on each trace, next to the linked prompt version.</p>
    </div>
  </div>
</div>

In [ ]:
displayHTML(
    f'<a href="https://{WORKSPACE_URL}/ml/experiments/{PROD_EXPERIMENT_ID}" target="_blank">'
    "Open the production experiment and click Traces</a>"
)

Because the traces are Delta tables, you can also read them with SQL, for example to build a dashboard or an alert. Each row of the spans table is one span.

In [ ]:
spans_table = f"{CATALOG}.{SCHEMA}.{TRACE_TABLE_PREFIX}_otel_spans"
print(f"{spark.table(spans_table).count()} spans in {spans_table}")
display(spark.table(spans_table).limit(10))

# Summary

We started with one question: **"I changed the prompt. Is the app better now? Can I ship it?"**

MLflow answered it by connecting everything to the prompt version:

<div class="mermaid">
flowchart LR
    P["Prompt version<br/>v1, v2"] --- T["Traces"]
    T --- E["Evaluation runs<br/>scores + rationale"]
    P --- A["Alias<br/>production"]
    A --- APP["Deployed app"]
    APP --- M["Monitoring results"]
    style P fill:#fdf0ea,stroke:#c2410c,stroke-width:2px
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>


| Stage | Step | What we used | What it gives you |
|---|---|---|---|
| Development | 1 to 3 | `register_prompt`, `load_prompt`, `@mlflow.trace`, `autolog()` | Versioned prompts, and a record of every request |
| Offline evaluation | 4 and 5 | `make_judge`, `mlflow.genai.evaluate` | Scores for each version on the same test data, with reasons |
| Release | 6 | `set_prompt_alias` | Switch and roll back without changing code |
| Deployment | 7 | Databricks Apps | A web app that loads the prompt by alias |
| Online monitoring | 8 | `register()`, `start()` | Automatic quality checks on real traffic |

# Next steps

- **Try v3**: Write v3, register it, evaluate it, and move the `production` alias if it wins. Check that the app picks it up without a redeploy
- **Learn more about prompts**: [Prompt Registry](https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/)
- **Learn more about evaluation**: [Evaluate and improve](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)
- **Learn more about monitoring**: [Production quality monitoring](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring)
- **Learn more about apps**: [Databricks Apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps)

# Cleanup (optional)

There are two levels of cleanup:

| Cell | What it does | When to use it |
|---|---|---|
| Stop the monitors | Stops and deletes the two monitors. Everything else stays. Runs only with `RUN_STOP_MONITORS = True` | When you're done with the demo but want to keep your work |
| Full reset | Deletes everything this notebook created | When you want to run the notebook again from a clean slate |

<div style="border-left: 4px solid #ef6c00; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;">⚠️</span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">The monitors and the app keep running</strong>
      <p style="margin: 8px 0 0 0; color: #333;">The monitors score new traces every 15 minutes until you stop them. On Free Edition, the app stops automatically 24 hours after it starts, but it stays in your workspace and counts toward the limit of 3 apps until you delete it.</p>
    </div>
  </div>
</div>

In [ ]:
# --- Stop the monitors ---
# Stop the background jobs, then delete the monitors from the production experiment.
# The app, the prompts, the experiments and the data stay as they are.
# Set this to True when you're done. It is False so that "Run all" doesn't remove the monitors.
RUN_STOP_MONITORS = False

if not RUN_STOP_MONITORS:
    print("Skipped. Set RUN_STOP_MONITORS = True to stop and delete the monitors.")
else:
    from mlflow.genai.scorers import delete_scorer, get_scorer

    for name in ["schema_compliance", "topics_relevance"]:
        try:
            get_scorer(name=name, experiment_id=PROD_EXPERIMENT_ID).stop(experiment_id=PROD_EXPERIMENT_ID)
            delete_scorer(name=name, experiment_id=PROD_EXPERIMENT_ID)
            print(f"Stopped and deleted {name}")
        except Exception as e:
            print(f"Skipped {name}: {e}")

## Full reset

This cell deletes everything this notebook created, in this order:

1. The monitors in the production experiment
2. The app (its service principal is deleted with it)
3. The app's code folder in your workspace
4. The prompt: its aliases, all its versions, then the prompt itself
5. The two experiments, with their traces and evaluation runs
6. The schema `workspace.genai_demo`, with the review table and the trace tables

It can't be undone. To run it, set `RUN_FULL_RESET = True`. After the reset, run the notebook again from Step 0.

In [ ]:
# --- Full reset ---
# Deletes everything this notebook created. This can't be undone.
# Set this to True only when you really want to start over.
RUN_FULL_RESET = False

if not RUN_FULL_RESET:
    print("Skipped. Set RUN_FULL_RESET = True to delete everything.")
else:
    import time

    from databricks.sdk import WorkspaceClient
    from mlflow import MlflowClient
    from mlflow.genai.scorers import delete_scorer, get_scorer

    client = MlflowClient()
    w = WorkspaceClient()
    APP_NAME = "genai-review-analyzer"
    APP_DIR = f"/Workspace/Users/{username}/genai_review_app"
    PROD_EXPERIMENT_NAME = f"/Users/{username}/genai_review_app_production"

    # 1. Monitors: stop the background jobs and delete them
    prod_experiment = mlflow.get_experiment_by_name(PROD_EXPERIMENT_NAME)
    if prod_experiment is not None:
        for name in ["schema_compliance", "topics_relevance"]:
            try:
                get_scorer(name=name, experiment_id=prod_experiment.experiment_id).stop(
                    experiment_id=prod_experiment.experiment_id
                )
                delete_scorer(name=name, experiment_id=prod_experiment.experiment_id)
                print(f"Deleted the monitor {name}")
            except Exception as e:
                print(f"Skipped the monitor {name}: {e}")

    # 2. The app. Its service principal is deleted with it.
    try:
        w.apps.delete(APP_NAME)
        print(f"Deleted the app {APP_NAME}")
    except Exception as e:
        print(f"Skipped the app: {e}")

    # 3. The app's code folder
    try:
        w.workspace.delete(APP_DIR, recursive=True)
        print(f"Deleted {APP_DIR}")
    except Exception as e:
        print(f"Skipped the app folder: {e}")

    # 4. The prompt. In Unity Catalog, you must delete the aliases and all versions first.
    for alias in ["production", "staging"]:
        try:
            mlflow.genai.delete_prompt_alias(name=PROMPT_NAME, alias=alias)
        except Exception:
            pass
    try:
        # Collect all versions first (the results come in pages), then delete them
        versions, token = [], None
        while True:
            page = client.search_prompt_versions(PROMPT_NAME, page_token=token)
            versions.extend(page)
            token = page.token
            if not token:
                break
        for version in versions:
            client.delete_prompt_version(PROMPT_NAME, version.version)
        client.delete_prompt(PROMPT_NAME)
        print(f"Deleted the prompt {PROMPT_NAME}")
    except Exception as e:
        print(f"Skipped the prompt: {e}")

    # 5. The experiments. A deleted experiment keeps its name, so we rename it first.
    #    This lets you create a new experiment with the same name when you run the notebook again.
    for exp_name in [f"/Users/{username}/genai_prototype_to_production", PROD_EXPERIMENT_NAME]:
        exp = mlflow.get_experiment_by_name(exp_name)
        if exp is None:
            continue
        client.rename_experiment(exp.experiment_id, f"{exp_name}_deleted_{int(time.time())}")
        client.delete_experiment(exp.experiment_id)
        print(f"Deleted the experiment {exp_name}")

    # 6. The schema, with the review table
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
    print(f"Dropped the schema {CATALOG}.{SCHEMA}")

    print("Full reset complete. Run the notebook again from Step 0.")